# 08 — Final submission v4
Retrain trên toàn bộ labeled train với Stratified 5-fold, cùng folds cho LGB/XGB. Lưu OOF và test prediction riêng từng model (kể cả khi config chọn single model), sau đó áp dụng config đã khóa từ NB06. OOF ở đây là diagnostic phát triển vì feature/params đã được chọn trước, không phải nested CV. Không dùng threshold để submit.

In [1]:
from pathlib import Path
import sys, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Run from project root or notebooks/"
sys.path.insert(0, str(PROJECT_ROOT))
from src.features import add_features, FEATURE_VERSION, FEATURE_SETS, CATEGORICAL_COLUMNS
from src.config import PRIMARY_METRIC as PRIMARY_METRIC_KEY, METRIC_LABELS, OUTPUT_VERSION
from src.split import time_based_split, create_stratified_kfold
assert FEATURE_VERSION == "v3", "Copy features_v3.py to src/features.py and restart kernel"
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = DATA_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
N_SPLITS = 5
N_JOBS = 4
# Use "cuda:0" on a machine with a compatible XGBoost GPU setup.
XGB_DEVICE = "cpu"

import joblib

In [2]:
def load_featured(kind):
    raw_path = DATA_DIR / f"{kind}.csv"
    cache_path = DATA_DIR / f"{kind}_featured_{FEATURE_VERSION}.pkl"
    required = set(FEATURE_SETS["compact_all"]) | {"id"}
    identity_cols = ["id", "is_fraud"] if kind == "train" else ["id"]
    required.update(identity_cols)
    if cache_path.exists() and cache_path.stat().st_mtime >= raw_path.stat().st_mtime:
        try:
            frame = pd.read_pickle(cache_path)
            identity = pd.read_csv(raw_path, usecols=identity_cols)
            if not required.issubset(frame.columns):
                raise ValueError("Missing v3 features")
            for col in identity_cols:
                if not np.array_equal(frame[col].to_numpy(), identity[col].to_numpy()):
                    raise ValueError(f"Cached {col} alignment mismatch")
            if not frame["id"].is_unique:
                raise ValueError("Duplicate IDs")
            print("Loaded:", cache_path.name)
            return frame.reset_index(drop=True)
        except (OSError, ImportError, AttributeError, TypeError, ValueError,
                NotImplementedError, EOFError, pickle.UnpicklingError) as exc:
            print(f"Rebuilding {cache_path.name}: {exc}")
    raw = pd.read_csv(raw_path)
    frame = add_features(raw.drop(columns="is_fraud", errors="ignore"))
    if kind == "train":
        frame["is_fraud"] = raw["is_fraud"].astype(int)
    assert frame["id"].is_unique and required.issubset(frame.columns)
    frame.to_pickle(cache_path)
    print("Saved:", cache_path.name)
    return frame.reset_index(drop=True)


def align_categories(train, *others, cat_cols):
    train = train.copy()
    others = [frame.copy() for frame in others]
    for col in cat_cols:
        # Materialize strings to ensure XGBoost-compatible categorical labels.
        train[col] = train[col].astype(object).astype("category").cat.remove_unused_categories()
        cats = train[col].cat.categories
        for frame in others:
            frame[col] = pd.Categorical(frame[col].astype(object), categories=cats)
    return (train, *others)


def split_development(frame):
    parts = time_based_split(frame, id_col="id", train_ratio=0.70,
                             valid_ratio=0.15, buffer_size=0)
    return tuple(parts[k].sort_values("id").reset_index(drop=True)
                 for k in ("train", "valid", "test_holdout"))


def scores(y, p):
    return {"ROC-AUC": float(roc_auc_score(y, p)),
            "AP": float(average_precision_score(y, p)),
            "Brier": float(brier_score_loss(y, p))}


def fixed_params(config):
    out = {"LightGBM": dict(config["lightgbm_params"]),
           "XGBoost": dict(config["xgboost_params"])}
    for params in out.values():
        params.pop("early_stopping_rounds", None)
        params.pop("callbacks", None)
        params["n_jobs"] = N_JOBS
    out["XGBoost"].update(device=XGB_DEVICE, enable_categorical=True, tree_method="hist")
    return out


def build_model(name, params):
    return (lgb.LGBMClassifier if name == "LightGBM" else xgb.XGBClassifier)(**params)

In [3]:
config_path = ARTIFACT_DIR / f"final_model_config_{OUTPUT_VERSION}.json"
config = json.loads(config_path.read_text(encoding="utf-8"))
assert config["feature_version"] == FEATURE_VERSION and config["output_version"] == OUTPUT_VERSION
assert config["optimization_metric"] == METRIC_LABELS[PRIMARY_METRIC_KEY] == "ROC-AUC"
FEATURE_COLS = config["feature_columns"]
CAT_COLS = config["categorical_columns"]
SELECTED = config["selected_model"]
assert FEATURE_COLS == FEATURE_SETS[config["feature_set"]], "Feature set changed; rerun NB05/06"
assert CAT_COLS == [c for c in FEATURE_COLS if c in CATEGORICAL_COLUMNS]
assert SELECTED in ("LightGBM", "XGBoost", "Weighted Blending")
RANDOM_STATE = int(config["random_state"])
N_SPLITS = int(config["n_splits"])
params = fixed_params(config)
a = float(config["ensemble_weights"]["lightgbm"])
assert 0 <= a <= 1 and np.isclose(a + config["ensemble_weights"]["xgboost"], 1)
print(SELECTED, config["feature_set"], len(FEATURE_COLS), config["optimization_metric"])


def select_prediction(preds):
    if SELECTED == "Weighted Blending":
        return a * preds["LightGBM"] + (1 - a) * preds["XGBoost"]
    return preds[SELECTED]

LightGBM raw_core 23 ROC-AUC


In [4]:
train_df = load_featured("train")
test_df = load_featured("test")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
SUBMIT_DIR = PROJECT_ROOT / "submissions"
SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
assert set(FEATURE_COLS).issubset(train_df.columns) and set(FEATURE_COLS).issubset(test_df.columns)
# Keep original project fold generator for champion comparability.
train_df = train_df.drop(columns="fold", errors="ignore")
train_df = create_stratified_kfold(train_df, target_col="is_fraud", n_splits=N_SPLITS,
                                   shuffle=True, random_state=RANDOM_STATE, fold_col="fold")
y_all = train_df.is_fraud.to_numpy(dtype=int)
folds = train_df.fold.to_numpy()
assert set(np.unique(folds)) == set(range(N_SPLITS))
train_ids = train_df.id.to_numpy()
test_ids = test_df.id.to_numpy()
X_all, X_test = train_df[FEATURE_COLS], test_df[FEATURE_COLS]
assert X_all.columns.equals(X_test.columns)
print(X_all.shape, X_test.shape)

Loaded: train_featured_v3.pkl
Loaded: test_featured_v3.pkl
(699960, 23) (299984, 23)


In [5]:
model_names = ("LightGBM", "XGBoost")
oof = {name: np.full(len(y_all), np.nan) for name in model_names}
test_preds = {name: np.zeros(len(X_test)) for name in model_names}
trained_models, fold_rows = [], []
SAVE_MODELS = True
for fold in range(N_SPLITS):
    tr, va = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
    X_tr, X_va, X_te = align_categories(X_all.iloc[tr], X_all.iloc[va], X_test, cat_cols=CAT_COLS)
    fitted, val_preds = {}, {}
    for name in model_names:
        model = build_model(name, params[name]).fit(X_tr, y_all[tr])
        p_va, p_te = model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]
        oof[name][va] = p_va
        val_preds[name] = p_va
        test_preds[name] += p_te / N_SPLITS
        fold_rows.append({"fold": fold, "model": name, **scores(y_all[va], p_va)})
        if SAVE_MODELS:
            fitted[name] = model
    p_selected = select_prediction(val_preds)
    fold_rows.append({"fold": fold, "model": "Selected", **scores(y_all[va], p_selected)})
    print(f"Fold {fold}", scores(y_all[va], p_selected))
    if SAVE_MODELS:
        trained_models.append({"fold": fold, "models": fitted,
                               "categories": {c: X_tr[c].cat.categories.tolist() for c in CAT_COLS}})
for p in [*oof.values(), *test_preds.values()]:
    assert np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()
oof_selected = select_prediction(oof)
predictions = select_prediction(test_preds)
fold_report = pd.DataFrame(fold_rows)
pooled_report = pd.DataFrame([{"model": name, **scores(y_all, p)} for name, p in {**oof, "Selected": oof_selected}.items()])
display(fold_report)
display(fold_report.groupby("model")[["AP", "ROC-AUC"]].agg(["mean", "std"]))
display(pooled_report)
fold_report.to_csv(ARTIFACT_DIR / f"08_fold_metrics_{OUTPUT_VERSION}.csv", index=False)
pooled_report.to_csv(ARTIFACT_DIR / f"08_pooled_oof_metrics_{OUTPUT_VERSION}.csv", index=False)
np.savez_compressed(ARTIFACT_DIR / f"08_oof_test_predictions_{OUTPUT_VERSION}.npz",
    train_ids=np.asarray(train_ids, dtype=str), test_ids=np.asarray(test_ids, dtype=str), y_true=y_all, folds=folds,
    oof_lgb=oof["LightGBM"], oof_xgb=oof["XGBoost"], oof_selected=oof_selected,
    test_pred_lgb=test_preds["LightGBM"], test_pred_xgb=test_preds["XGBoost"], test_pred_selected=predictions)

Fold 0 {'ROC-AUC': 0.94373977181458, 'AP': 0.49795159254333055, 'Brier': 0.011741327021185154}
Fold 1 {'ROC-AUC': 0.9456657536482382, 'AP': 0.5148488921236772, 'Brier': 0.011434775221211889}
Fold 2 {'ROC-AUC': 0.9487945582757716, 'AP': 0.4989658231005577, 'Brier': 0.011644389328353417}
Fold 3 {'ROC-AUC': 0.9467333067250859, 'AP': 0.5192321534093612, 'Brier': 0.011279197529133663}
Fold 4 {'ROC-AUC': 0.9453648518475117, 'AP': 0.5064578137713123, 'Brier': 0.011546738162999242}


,fold,model,ROC-AUC,AP,Brier
0,0,LightGBM,0.943740,0.497952,0.011741
1,0,XGBoost,0.943401,0.497104,0.011652
2,0,Selected,0.943740,0.497952,0.011741
3,1,LightGBM,0.945666,0.514849,0.011435
4,1,XGBoost,0.946010,0.512454,0.011360
5,1,Selected,0.945666,0.514849,0.011435
6,2,LightGBM,0.948795,0.498966,0.011644
7,2,XGBoost,0.948494,0.498401,0.011547
8,2,Selected,0.948795,0.498966,0.011644
9,3,LightGBM,0.946733,0.519232,0.011279


AP             ROC-AUC          
              mean       std      mean       std
model                                           
LightGBM  0.507491  0.009444  0.946060  0.001868
Selected  0.507491  0.009444  0.946060  0.001868
XGBoost   0.506366  0.008812  0.945998  0.001841

,model,ROC-AUC,AP,Brier
0,LightGBM,0.946049,0.507178,0.011529
1,XGBoost,0.945984,0.506039,0.011445
2,Selected,0.946049,0.507178,0.011529


In [6]:
target = "is_fraud"
assert target in sample.columns
assert len(predictions) == len(test_ids) and np.isfinite(predictions).all()
assert ((predictions >= 0) & (predictions <= 1)).all()
submission = sample.copy()
if "id" in sample.columns:
    assert sample.id.is_unique and pd.Index(test_ids).is_unique
    assert len(sample) == len(test_ids) and set(sample.id) == set(test_ids)
    submission[target] = submission.id.map(pd.Series(predictions, index=test_ids))
else:
    assert len(sample) == len(predictions)
    submission[target] = predictions
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[target].notna().all() and submission[target].between(0, 1).all()
# Unique filename, preserving previous submissions even when running the same feature set.
from datetime import datetime
run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
output = SUBMIT_DIR / f"submission_{OUTPUT_VERSION}_{config['feature_set']}_{run_id}.csv"
submission.to_csv(output, index=False)
saved = pd.read_csv(output)
assert saved.shape == sample.shape and saved.columns.tolist() == sample.columns.tolist()
if "id" in sample.columns:
    assert saved.id.equals(sample.id)
print("Saved:", output)
display(submission.head())

Saved: D:\Project\FraudDetection\submissions\submission_v4_raw_core_20261124_091205_007956.csv


,id,is_fraud
0,699960,0.000406
1,699961,0.000254
2,699962,0.142049
3,699963,0.002477
4,699964,0.002224


In [7]:
if SAVE_MODELS:
    joblib.dump({"models": trained_models, "config": config, "features": FEATURE_COLS,
                 "categoricals": CAT_COLS, "actual_xgb_device": XGB_DEVICE},
                 ARTIFACT_DIR / f"08_final_fitted_models_{OUTPUT_VERSION}.joblib")
metadata = {"feature_version": FEATURE_VERSION, "output_version": OUTPUT_VERSION, "feature_set": config["feature_set"],
            "selected_model": SELECTED, "train_rows": len(y_all), "test_rows": len(test_ids),
            "submission": output.name, "prediction_min": float(predictions.min()),
            "prediction_max": float(predictions.max()), "n_splits": N_SPLITS,
            "actual_xgb_device": XGB_DEVICE, "pooled_oof": scores(y_all, oof_selected)}
(ARTIFACT_DIR / f"08_submission_metadata_{OUTPUT_VERSION}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Verified submission:", output)

Verified submission: D:\Project\FraudDetection\submissions\submission_v4_raw_core_20261124_091205_007956.csv
